In [1]:
# Auto-generated by build_and_verify.py — do not hand-edit.
# Rebuild with: python build_and_verify.py lesson_20_free_energy_and_phase_equilibria.yaml

from cs470_engine import Worksheet
from cs470_engine.plot_style import apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_20_free_energy_and_phase_equilibria.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 20: Free Energy Calculations, Phase Equilibria & Flat-Histogram Methods

_Session e5d0958f · Estimated time: ~120 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Explain why free energy is an entropic phase-space volume property ($F = -k_B T \ln Z$) rather than a simple mechanical ensemble average.
- Derive the Kirkwood coupling-constant Thermodynamic Integration (TI) formula and evaluate path-dependent hysteresis across first-order phase transitions.
- Analyze Zwanzig Free Energy Perturbation (FEP) and diagnose the unidirectional phase-space overlap catastrophe.
- Implement Widom test-particle insertion for excess chemical potential and identify its exponential breakdown at dense liquid and solid packing.
- Formulate the Gibbs Ensemble Monte Carlo (GEMC) technique for direct phase coexistence without explicit physical interfaces.
- Derive the Bennett Acceptance Ratio (BAR) minimum-variance estimator from the Crooks fluctuation theorem.
- Understand the Wang-Landau flat-histogram algorithm for microcanonical density of states $g(E)$ and microcanonical entropy $S(E)$.


In [2]:
ws.render_section('sec_free_energy_foundations')

### The Free Energy Bottleneck

Mechanical observables such as internal energy $E = \langle \mathcal{H} \rangle$, pressure $P = \rho k_B T + \frac{1}{3V}\langle \sum \mathbf{r}_{ij}\cdot\mathbf{f}_{ij} \rangle$, or pair correlation $g(r)$ can be computed directly as arithmetic averages along a simulation trajectory. In contrast, Helmholtz free energy $F$ is defined by:

$$F = -k_B T \ln Q(N, V, T), \qquad Q = \frac{1}{N! \Lambda^{3N}} \int d\mathbf{r}^N e^{-\beta V(\mathbf{r}^N)}$$

Free energy is proportional to the **logarithm of the partition function** $Q$, which measures the total hypervolume of accessible phase space. Standard Monte Carlo and MD trajectories sample states according to the Boltzmann probability $P(\mathbf{r}^N) \propto e^{-\beta V(\mathbf{r}^N)}$; they never explore the absolute normalisation factor $Q$.

### Zwanzig Free Energy Perturbation (FEP)

To measure the free energy difference $\Delta F = F_1 - F_0$ between two systems with potential energies $V_0$ and $V_1 = V_0 + \Delta V$, Zwanzig (1954) formulated the exact identity:

$$e^{-\beta \Delta F} = \frac{Q_1}{Q_0} = \frac{\int d\mathbf{r}^N e^{-\beta V_1}}{\int d\mathbf{r}^N e^{-\beta V_0}} = \frac{\int d\mathbf{r}^N e^{-\beta \Delta V} e^{-\beta V_0}}{\int d\mathbf{r}^N e^{-\beta V_0}} = \left\langle e^{-\beta \Delta V} \right\rangle_0$$

While mathematically exact, FEP suffers from a severe **phase-space overlap problem**: if states with low energy in system 1 correspond to rare, high-energy configurations in system 0, the ensemble average $\langle e^{-\beta \Delta V} \rangle_0$ is dominated by extreme, rarely sampled tail events where $\Delta V \ll 0$, leading to enormous statistical errors and systematic bias.


In [3]:
ws.render_section('sec_thermodynamic_integration_kirkwood')

### Coupling-Constant Thermodynamic Integration

Rather than attempting a large single perturbation, Kirkwood (1935) introduced an alchemical coupling parameter $\lambda \in [0, 1]$ connecting system 0 ($V_0$) to system 1 ($V_1$):

$$V_\lambda(\mathbf{r}^N) = (1 - \lambda) V_0(\mathbf{r}^N) + \lambda V_1(\mathbf{r}^N)$$

Differentiating the free energy $F_\lambda = -k_B T \ln Q_\lambda$ with respect to $\lambda$ yields:

$$\frac{\partial F_\lambda}{\partial \lambda} = -\frac{k_B T}{Q_\lambda} \frac{\partial Q_\lambda}{\partial \lambda} = \frac{\int d\mathbf{r}^N \left( \frac{\partial V_\lambda}{\partial \lambda} \right) e^{-\beta V_\lambda}}{\int d\mathbf{r}^N e^{-\beta V_\lambda}} = \left\langle \frac{\partial V_\lambda}{\partial \lambda} \right\rangle_\lambda$$

For linear coupling, $\partial V_\lambda / \partial \lambda = V_1 - V_0 = \Delta V$. Integrating along $\lambda$ from 0 to 1 gives the exact free energy difference:

$$\Delta F = F_1 - F_0 = \int_0^1 \left\langle \frac{\partial V_\lambda}{\partial \lambda} \right\rangle_\lambda d\lambda = \int_0^1 \langle V_1 - V_0 \rangle_\lambda d\lambda$$

### Convexity and Path-Crossing Hysteresis

Differentiating a second time reveals that the free energy is strictly concave with respect to $\lambda$:

$$\frac{\partial^2 F_\lambda}{\partial \lambda^2} = -\beta \left[ \left\langle \left( \frac{\partial V_\lambda}{\partial \lambda} \right)^2 \right\rangle_\lambda - \left\langle \frac{\partial V_\lambda}{\partial \lambda} \right\rangle_\lambda^2 \right] = -\beta \,\mathrm{Var}_\lambda\left(\frac{\partial V_\lambda}{\partial \lambda}\right) \le 0$$

Therefore, the integrand $\langle \Delta V \rangle_\lambda$ is a monotonically decreasing function of $\lambda$.

If the alchemical path accidentally crosses a **first-order phase transition** (e.g. solid-liquid coexistence), the system superheats or supercools, producing strong **hysteresis loops** between forward ($0 \to 1$) and reverse ($1 \to 0$) sweeps. To obtain accurate free energies across phases, one must avoid phase boundaries by integrating along reversible paths (such as the Frenkel-Ladd Einstein crystal path).


In [4]:
ws.render_section('concept_kirkwood_thermodynamic_integration')

### Interactive Concept: Kirkwood Thermodynamic Integration & Hysteresis

Explore Kirkwood coupling-parameter integration $\Delta F = \int_0^1 \langle \Delta V \rangle_\lambda d\lambda$.
Adjust quadrature points, anharmonic strength, and switch between a smooth single-phase path and a path crossing a first-order phase transition.

**Key Insight:** Along a reversible path, $\langle \Delta V \rangle_\lambda$ varies smoothly and a 6-to-8 point Gauss-Legendre quadrature yields high accuracy. When crossing a first-order phase boundary, nucleation barriers trap the system in metastable states, causing forward and backward sweeps to diverge and dissipate irreversible hysteresis work.

In [5]:
ws.render_section('q20_01')

**Q1 · easy** · 💡

Why cannot the Helmholtz free energy $F = -k_B T \ln Q$ be evaluated simply by averaging a microscopic mechanical variable $\langle A(\mathbf{r}^N) \rangle$ in standard Monte Carlo or Molecular Dynamics?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what the partition function $Q$ represents mathematically in phase space.

</div>
</details>

- [ ] **(A)** Free energy depends on quantum spin operators that are neglected in classical MD.
- [ ] **(B)** Free energy is an entropic, phase-space volume property determined by the normalization integral $Q$, whereas importance sampling only explores relative probabilities $P \propto e^{-\beta V}$ without computing $Q$.
- [ ] **(C)** Free energy is only defined for ideal gases.
- [ ] **(D)** The virial theorem proves that free energy is identical to kinetic energy.

In [6]:
ws.render_section('q20_02')

**Q2 · medium** · 💡

In the Zwanzig Free Energy Perturbation (FEP) formula $e^{-\beta \Delta F} = \langle e^{-\beta \Delta V} \rangle_0$, which ensemble is used to generate configurations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice the subscript 0 on the expectation brackets $\langle \dots \rangle_0$.

</div>
</details>

- [ ] **(A)** The target ensemble 1 (with potential $V_1$)
- [ ] **(B)** The reference ensemble 0 (with potential $V_0$)
- [ ] **(C)** An unweighted microcanonical ensemble where all energies are equal
- [ ] **(D)** A grand canonical ensemble with chemical potential $\mu = 0$

In [7]:
ws.render_section('q20_03')

**Q3 · medium** · 💡

Why does forward FEP systematically fail when the target state 1 has a significantly smaller phase-space volume than reference state 0 (e.g., creating a hard sphere cavity in a dense fluid)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Where do the most important configurations for state 1 lie in the probability distribution of state 0?

</div>
</details>

- [ ] **(A)** Because the kinetic energy in state 1 is negative.
- [ ] **(B)** Because the configurations that contribute most heavily to $\langle e^{-\beta \Delta V} \rangle_0$ (where $\Delta V \ll 0$) are virtually never sampled in ensemble 0.
- [ ] **(C)** Because $e^{-\beta \Delta V}$ diverges to infinity for all configurations.
- [ ] **(D)** Because Newton's third law is violated.

In [8]:
ws.render_section('q20_04')

**Q4 · easy** · 💡

What is the integrand in Kirkwood's alchemical thermodynamic integration $\Delta F = \int_0^1 I(\lambda) d\lambda$ for a linearly coupled potential $V_\lambda = (1-\lambda)V_0 + \lambda V_1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate $V_\lambda$ with respect to $\lambda$: $\partial V_\lambda / \partial \lambda = V_1 - V_0$.

</div>
</details>

- [ ] **(A)** $I(\lambda) = \langle e^{-\beta (V_1 - V_0)} \rangle_\lambda$
- [ ] **(B)** $I(\lambda) = \langle V_1 - V_0 \rangle_\lambda$
- [ ] **(C)** $I(\lambda) = \langle V_0 \rangle_\lambda \cdot \langle V_1 \rangle_\lambda$
- [ ] **(D)** $I(\lambda) = \beta \langle V_1 + V_0 \rangle_\lambda$

In [9]:
ws.render_section('q20_05')

**Q5 · hard** · 💡

Prove the slope of the Kirkwood integrand: why must $\frac{d}{d\lambda}\langle V_1 - V_0 \rangle_\lambda \le 0$ for any physical system?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\frac{\partial^2 F_\lambda}{\partial \lambda^2}$ by differentiating the Boltzmann-weighted expectation value.

</div>
</details>

- [ ] **(A)** Because the derivative equals $-\beta \,\mathrm{Var}_\lambda(V_1 - V_0)$, and variance is strictly non-negative.
- [ ] **(B)** Because entropy always increases in an isolated system.
- [ ] **(C)** Because the coupling parameter $\lambda$ is bounded between 0 and 1.
- [ ] **(D)** Because the partition function $Q_\lambda$ is a linear function of $\lambda$.

In [10]:
ws.render_section('q20_06')

**Q6 · medium** · 💡

A 3-point Simpson's rule is used for Kirkwood TI: $\Delta F \approx \frac{1}{6}[I(0) + 4 I(0.5) + I(1.0)]$. If the measured values are $I(0) = 12.0\text{ kcal/mol}$, $I(0.5) = 7.5\text{ kcal/mol}$, and $I(1.0) = 4.0\text{ kcal/mol}$, calculate $\Delta F$ in kcal/mol.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\frac{1}{6}[12.0 + 4(7.5) + 4.0]$.

</div>
</details>

_Numeric answer required (in kcal/mol)._

In [11]:
ws.render_section('q20_07')

**Q7 · medium** · 💡

In the concept cell, when switching to 'First-Order Transition Crossing', what causes the forward sweep ($\\lambda: 0 \\to 1$) and reverse sweep ($\\lambda: 1 \\to 0$) to form a hysteresis loop?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall what happens when water is rapidly cooled below 0 °C without ice nuclei.

</div>
</details>

- [ ] **(A)** The computer uses single precision instead of double precision.
- [ ] **(B)** A first-order transition features a free energy barrier between distinct phases, trapping the system in metastable states (superheating and supercooling) during finite-rate sweeps.
- [ ] **(C)** The Kirkwood formula is mathematically invalid for any system with interactions.
- [ ] **(D)** The coupling parameter $\lambda$ becomes imaginary.

In [12]:
ws.render_section('q20_08')

**Q8 · medium** · 💡

How did Frenkel and Ladd solve the problem of phase-crossing hysteresis when calculating the absolute free energy of a crystalline solid?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What reference system has an analytically solvable partition function for localized particles?

</div>
</details>

- [ ] **(A)** They melted the crystal into a liquid and integrated along the boiling curve.
- [ ] **(B)** They constructed a reversible thermodynamic path connecting the interacting solid to an 'Einstein crystal' (independent harmonic oscillators anchored to lattice sites), whose absolute free energy is known analytically.
- [ ] **(C)** They set the temperature of the crystal to absolute zero ($T = 0\text{ K}$).
- [ ] **(D)** They replaced all atoms with ideal gas particles in a single step.

In [13]:
ws.render_section('q20_09')

**Q9 · easy** · 💡

Can thermodynamic integration be performed using physical state variables like temperature $T$ or volume $V$ instead of an alchemical parameter $\lambda$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the fundamental thermodynamic relation $dF = -S dT - P dV$ and the Gibbs-Helmholtz equation.

</div>
</details>

- [ ] **(A)** No, TI is strictly defined only for fictitious coupling parameters $\lambda$.
- [ ] **(B)** Yes: integrating pressure gives $\Delta F = -\int_{V_0}^{V_1} P(V) dV$ at constant $T$, and integrating energy gives $\Delta(F/T) = -\int_{T_0}^{T_1} \frac{E(T)}{T^2} dT$ at constant $V$.
- [ ] **(C)** Yes, but only if the temperature is negative.
- [ ] **(D)** No, because $P$ and $E$ are not thermodynamic state functions.

In [14]:
ws.render_section('q20_10')

**Q10 · medium** · 💡

Which of the following practices ensure accurate numerical evaluation of $\int_0^1 \langle \Delta V \rangle_\lambda d\lambda$? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about quadrature theory, equilibration, and what happens when an atom appears on top of another atom at small $\lambda$.

</div>
</details>

- [ ] **(A)** Use Gauss-Legendre quadrature nodes rather than equally spaced trapezoidal points to maximize polynomial accuracy.
- [ ] **(B)** Equilibrate the system thoroughly at each $\lambda$ node before accumulating the ensemble average $\langle \Delta V \rangle_\lambda$.
- [ ] **(C)** Place a single integration point at $\lambda = 0.5$ and assume linearity.
- [ ] **(D)** Use 'soft-core' potentials when turning on repulsive atom centers to prevent the $r^{-12}$ singularity at $\lambda \to 0$.

In [15]:
ws.render_section('sec_widom_and_phase_equilibria')

### Chemical Potential and Widom Ghost Insertion

The chemical potential $\mu = (\partial F / \partial N)_{V, T}$ consists of an ideal gas contribution and an excess contribution:

$$\mu = \mu_{\text{id}} + \mu_{\text{ex}}, \qquad \mu_{\text{id}} = k_B T \ln(\rho \Lambda^3)$$

Widom (1963) showed that the excess chemical potential $\mu_{\text{ex}}$ is given by inserting a fictitious **ghost particle** at a random uniform position $\mathbf{s}$ in an $N$-particle fluid:

$$\mu_{\text{ex}} = -k_B T \ln \left\langle \int d\mathbf{s} \, e^{-\beta \Delta U_{\text{test}}(\mathbf{s})} \right\rangle_N = -k_B T \ln \left\langle e^{-\beta \Delta U_{\text{test}}} \right\rangle_N$$

where $\Delta U_{\text{test}}$ is the interaction energy between the test particle and the existing $N$ particles. Crucially, the test particle does not disturb the trajectory—it is purely a virtual probe.

### Breakdown at High Density

In dilute gases and low-density liquids, test particles frequently find empty cavities, giving non-zero Boltzmann factors $e^{-\beta \Delta U} > 0$. However, as density increases toward the liquid-solid boundary (packing fraction $\eta = \frac{\pi}{6}\rho\sigma^3 > 0.45$), the probability of inserting a sphere without overlapping an existing hard core drops exponentially: $P_{\text{cavity}} \sim e^{-c \eta / (1-\eta)^3}$. Virtually every insertion encounters hard core overlap ($\Delta U \gg 0 \implies e^{-\beta \Delta U} = 0$), causing Widom insertion to collapse completely.

### Gibbs Ensemble Monte Carlo (GEMC)

Determining vapor-liquid coexistence historically required simulating a massive slab containing an explicit liquid-gas interface. Panagiotopoulos (1987) devised the **Gibbs Ensemble**, which simulates two separate macroscopic boxes (Box 1 and Box 2) without any interface, governed by three coupled trial moves:

1. **Displacement moves:** Standard Metropolis particle translations inside each box to maintain internal thermal equilibrium ($T_1 = T_2 = T$).
2. **Volume exchange moves:** $\Delta V_1 = -\Delta V_2$ at constant total volume $V = V_1 + V_2$, driving pressure equality ($P_1 = P_2$).
3. **Particle transfer moves:** Deleting a particle from Box 1 and inserting into Box 2, driving chemical potential equality ($\mu_1 = \mu_2$).

The two boxes spontaneously separate into pure saturated vapor and pure saturated liquid!


In [16]:
ws.render_section('concept_widom_and_gibbs_coexistence')

### Interactive Concept: Widom Ghost Insertion vs. Gibbs Ensemble Coexistence

Investigate the breakdown of Widom test-particle insertion at dense packing and observe Gibbs Ensemble coexistence.
Switch between the `widom` insertion efficiency plot and the `gibbs` two-box coexistence simulation.

**Key Insight:** Widom insertion works exceptionally well in gas and low-density fluid phases, but its insertion acceptance rate drops by 6 to 10 orders of magnitude in dense liquids due to core overlaps. The Gibbs Ensemble avoids this bottleneck by transferring volume and particles between two coupled, interface-free boxes to achieve thermodynamic phase coexistence.

In [17]:
ws.render_section('q20_11')

**Q11 · easy** · 💡

In Widom's test particle method, does the inserted test particle exert forces on the surrounding real particles and alter their trajectories?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice why Widom insertion is also called 'ghost particle insertion'.

</div>
</details>

- [ ] **(A)** Yes, it causes surrounding particles to recoil violently.
- [ ] **(B)** No, it is a purely virtual 'ghost' probe that evaluates the potential energy of interaction without affecting the actual system trajectory or coordinates.
- [ ] **(C)** Yes, but only if the temperature is above the critical temperature.
- [ ] **(D)** It permanently changes the total number of particles from $N$ to $N+1$.

In [18]:
ws.render_section('q20_12')

**Q12 · medium** · 💡

What happens to the insertion acceptance rate and statistical accuracy of the Widom method as the density of a hard-sphere fluid approaches dense liquid packing ($\rho \sigma^3 \approx 0.85$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What is the volume of empty cavities large enough to accommodate a whole sphere in a dense packing?

</div>
</details>

- [ ] **(A)** The acceptance rate approaches 100% and accuracy reaches machine precision.
- [ ] **(B)** The cavity insertion probability drops exponentially close to zero ($P_{\text{insert}} < 10^{-6}$), because almost all random insertion points overlap existing particle cores, causing the estimator to fail completely.
- [ ] **(C)** The chemical potential becomes negative infinity.
- [ ] **(D)** Widom insertion converts automatically into an MD simulation.

In [19]:
ws.render_section('q20_13')

**Q13 · medium** · 💡

In a Widom insertion simulation at $T = 300\text{ K}$ ($k_B T = 2.479\text{ kJ/mol}$), the ensemble average of the test-particle Boltzmann factor is measured to be $\langle e^{-\beta \Delta U} \rangle = 0.0183$. Calculate the excess chemical potential $\mu_{\text{ex}}$ in kJ/mol.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use $\mu_{\text{ex}} = -k_B T \ln \langle e^{-\beta \Delta U} \rangle$.

</div>
</details>

_Numeric answer required (in kJ/mol)._

In [20]:
ws.render_section('q20_14')

**Q14 · medium** · 💡

Which of the following conditions must be satisfied simultaneously for two phases (1 and 2) to be in thermodynamic equilibrium? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review classical thermodynamics criteria for phase equilibrium between two open, flexible subsystems.

</div>
</details>

- [ ] **(A)** Thermal equilibrium: $T_1 = T_2$
- [ ] **(B)** Mechanical equilibrium: $P_1 = P_2$
- [ ] **(C)** Chemical equilibrium: $\mu_1 = \mu_2$ for each component
- [ ] **(D)** Volume equality: $V_1 = V_2$
- [ ] **(E)** Particle number equality: $N_1 = N_2$

In [21]:
ws.render_section('q20_15')

**Q15 · medium** · 💡

In Gibbs Ensemble Monte Carlo (GEMC), how is mechanical equilibrium ($P_1 = P_2$) achieved between the two boxes?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How do subsystems exchange volume to equalize pressure in Gibbs ensemble?

</div>
</details>

- [ ] **(A)** By moving particles through a physical membrane between the boxes.
- [ ] **(B)** By trial volume exchanges $\Delta V_1 = -\Delta V_2$ with total volume $V_1 + V_2$ held constant, accepted according to the Metropolis rule involving pressure and volume ratios.
- [ ] **(C)** By adjusting the temperature of Box 1 while keeping Box 2 fixed.
- [ ] **(D)** By setting the external force on all atoms to zero.

In [22]:
ws.render_section('q20_16')

**Q16 · medium** · 💡

What major practical advantage does the Gibbs Ensemble offer over traditional direct-coexistence simulations of a vapor-liquid interface?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Why is an explicit interface computationally expensive in a small periodic box?

</div>
</details>

- [ ] **(A)** It eliminates the need for computing interatomic forces.
- [ ] **(B)** It avoids simulating the physical liquid-gas interface and its associated interfacial free energy, allowing small bulk systems ($N \sim 500$) to yield precise bulk coexistence densities without finite-size interfacial distortion.
- [ ] **(C)** It works at absolute zero temperature.
- [ ] **(D)** It guarantees that no particles ever change position.

In [23]:
ws.render_section('q20_17')

**Q17 · hard** · 💡

Why cannot the standard Gibbs Ensemble method be readily applied to study solid-liquid coexistence (e.g. melting of ice)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the particle transfer step of GEMC in an ordered crystalline lattice.

</div>
</details>

- [ ] **(A)** Because solids have zero entropy.
- [ ] **(B)** Because inserting or deleting whole atoms into a dense crystalline lattice without creating severe vacancy/interstitial defects has an astronomically low acceptance probability.
- [ ] **(C)** Because solids cannot be simulated with Monte Carlo.
- [ ] **(D)** Because the pressure in a solid is always negative.

In [24]:
ws.render_section('q20_18')

**Q18 · medium** · 💡

In the concept cell, in the 'widom' view, how does the insertion efficiency $P_{\text{accept}}$ behave as reduced density $\rho^*$ increases from 0.1 to 0.85?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Observe the logarithmic scale on the insertion probability curve in the Concept Cell.

</div>
</details>

- [ ] **(A)** It increases linearly.
- [ ] **(B)** It remains strictly constant at 50%.
- [ ] **(C)** It decays precipitously by several orders of magnitude (from $\sim 0.7$ down to $< 10^{-4}$), mirroring the vanishing fraction of available cavity space.
- [ ] **(D)** It oscillates periodically.

In [25]:
ws.render_section('q20_19')

**Q19 · easy** · 💡

In Grand Canonical Monte Carlo (GCMC), what state variables are held fixed?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the definition of the grand canonical ensemble $(\mu, V, T)$ from statistical mechanics.

</div>
</details>

- [ ] **(A)** Chemical potential $\mu$, Volume $V$, and Temperature $T$
- [ ] **(B)** Particle number $N$, Pressure $P$, and Enthalpy $H$
- [ ] **(C)** Energy $E$, Volume $V$, and Number $N$
- [ ] **(D)** Temperature $T$, Pressure $P$, and Entropy $S$

In [26]:
ws.render_section('q20_20')

**Q20 · hard** · 💡

For an ideal gas at density $\rho = 0.05\text{ nm}^{-3}$, what is the exact excess chemical potential $\mu_{\text{ex}}$ in units of $k_B T$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In an ideal gas, particles do not interact, so $\Delta U_{\text{test}} = 0$ everywhere.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('sec_bennett_acceptance_ratio')

### Optimal Bidirectional Sampling

Zwanzig FEP uses either forward sampling ($0 \to 1$) or backward sampling ($1 \to 0$). Bennett (1976) proved that the statistically optimal estimator that minimizes the asymptotic variance of $\Delta F$ combines independent samples from *both* ensembles:

Let $n_0$ configurations be sampled from ensemble 0 ($V_0$) and $n_1$ configurations from ensemble 1 ($V_1$). The Bennett Acceptance Ratio (BAR) self-consistent equation is:

$$\sum_{i=1}^{n_0} f(V_1^{(i)} - V_0^{(i)} - C) = \sum_{j=1}^{n_1} f(V_0^{(j)} - V_1^{(j)} + C)$$

where $f(x) = \frac{1}{1 + e^{\beta x}}$ is the **Fermi function**, and the optimal constant is:

$$C = \Delta F + \frac{1}{\beta} \ln\left( \frac{n_0}{n_1} \right)$$

Solving this equation iteratively for $\Delta F$ yields the minimum possible statistical error among all unbiased estimators.

### Fluctuation Theorems and Crooks Relation

Let $\rho_0(\Delta V)$ be the probability distribution of energy difference $\Delta V = V_1 - V_0$ in ensemble 0, and $\rho_1(\Delta V)$ be the corresponding distribution in ensemble 1. By exact Boltzmann reweighting:

$$\frac{\rho_0(\Delta V)}{\rho_1(\Delta V)} = \frac{Q_1}{Q_0} e^{\beta \Delta V} = e^{\beta (\Delta V - \Delta F)}$$

Taking the logarithm gives the famous **Crooks Fluctuation Relation**:

$$\ln \left[ \frac{\rho_0(\Delta V)}{\rho_1(\Delta V)} \right] = \beta (\Delta V - \Delta F)$$

Notice that at the exact crossing point where the two distributions intersect ($\rho_0(\Delta V) = \rho_1(\Delta V)$), the energy difference is identically equal to the free energy difference: $\Delta V^* = \Delta F$!


In [28]:
ws.render_section('concept_bennett_acceptance_overlap')

### Interactive Concept: Bennett Acceptance Ratio & Distribution Overlap

Visualize the forward $\rho_0(\Delta V)$ and reverse $\rho_1(\Delta V)$ energy difference distributions.
Adjust the perturbation shift, sample ratio $\ln(n_1/n_0)$, and switch between the distribution overlap and Fermi weighting function.

**Key Insight:** As long as $\rho_0$ and $\rho_1$ have non-zero overlap, the Bennett estimator bridges the two states with minimal variance. The crossing point where $\rho_0(\Delta V) = \rho_1(\Delta V)$ identifies the free energy difference $\Delta F$ directly. If the perturbation shift is too large, the distributions separate, overlap vanishes, and intermediate alchemical stratification states must be inserted.

In [29]:
ws.render_section('q20_21')

**Q21 · medium** · 💡

What mathematical function serves as the optimal weighting function $f(x)$ in the Bennett Acceptance Ratio (BAR) method?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the solution to the calculus of variations problem minimizing the variance of the free energy estimator.

</div>
</details>

- [ ] **(A)** Gaussian function $f(x) = e^{-x^2}$
- [ ] **(B)** Fermi-Dirac function $f(x) = \frac{1}{1 + e^{\beta x}}$
- [ ] **(C)** Dirac delta function $f(x) = \delta(x)$
- [ ] **(D)** Heaviside step function $f(x) = \Theta(x)$

In [30]:
ws.render_section('q20_22')

**Q22 · medium** · 💡

According to the Crooks Fluctuation Theorem $\frac{\rho_0(\Delta V)}{\rho_1(\Delta V)} = e^{\beta(\Delta V - \Delta F)}$, at what value of $\Delta V$ do the forward and backward distributions intersect ($\rho_0 = \rho_1$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Set $\rho_0(\Delta V) = \rho_1(\Delta V)$ so that the ratio equals 1.

</div>
</details>

- [ ] **(A)** At $\Delta V = 0$
- [ ] **(B)** At $\Delta V = \Delta F$ (the exact free energy difference)
- [ ] **(C)** At $\Delta V = \infty$
- [ ] **(D)** At $\Delta V = -k_B T$

In [31]:
ws.render_section('q20_23')

**Q23 · easy** · 💡

Why is BAR vastly superior to running two separate unidirectional FEP calculations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about variance reduction and utilizing all available simulation data from both end states.

</div>
</details>

- [ ] **(A)** BAR is non-iterative and requires zero floating-point operations.
- [ ] **(B)** BAR achieves the minimum theoretical variance by combining forward and reverse transition data, self-consistently canceling unilateral sampling biases.
- [ ] **(C)** BAR does not require evaluating energy differences.
- [ ] **(D)** BAR only works when no configurations overlap.

In [32]:
ws.render_section('q20_24')

**Q24 · medium** · 💡

In a BAR calculation with equal samples ($n_0 = n_1$), a single energy difference observation in ensemble 0 gives $\beta(V_1 - V_0 - \Delta F) = 0.0$. What is the Fermi weight $f(0) = \frac{1}{1 + e^0}$ assigned to this configuration?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $1 / (1 + 1)$.

</div>
</details>

_Numeric answer required._

In [33]:
ws.render_section('q20_25')

**Q25 · hard** · 💡

The Multistate Bennett Acceptance Ratio (MBAR / Shirts & Chodera, 2008) generalizes BAR to $K$ distinct thermodynamic states. What is the fundamental advantage of MBAR over standard pairwise BAR across a chain of $\lambda$ states?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how MBAR reweights samples from every simulated $\lambda_k$ window to all other windows.

</div>
</details>

- [ ] **(A)** MBAR uses configurations from all $K$ simulated states simultaneously to compute the free energy of every state, utilizing all sampled data globally and achieving lower variance than pairwise BAR.
- [ ] **(B)** MBAR eliminates the need to sample intermediate states altogether.
- [ ] **(C)** MBAR does not require Boltzmann weights.
- [ ] **(D)** MBAR only works for ideal gases.

In [34]:
ws.render_section('q20_26')

**Q26 · medium** · 💡

In the concept cell 'Bennett Acceptance Ratio & Overlap', what happens to the overlap integral when the perturbation separation $\Delta V$ increases from 1.0 to 6.0 $k_B T$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the shaded purple region between the two distributions as you drag the separation slider.

</div>
</details>

- [ ] **(A)** The overlap increases to 100%.
- [ ] **(B)** The overlap drops towards zero, and the statistical uncertainty in $\Delta F$ diverges because neither ensemble samples configurations near the crossing point.
- [ ] **(C)** The Fermi function becomes negative.
- [ ] **(D)** The two distributions merge into a single delta peak.

In [35]:
ws.render_section('q20_27')

**Q27 · easy** · 💡

What condition must be satisfied between two thermodynamic states for BAR to yield an accurate free energy difference without intermediate windows?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Can an estimator bridge two states if neither state ever visits configurations accessible to the other?

</div>
</details>

- [ ] **(A)** They must have identical volumes.
- [ ] **(B)** There must be significant phase-space overlap: configurations sampled in state 0 must have reasonable probability of occurring in state 1, and vice versa.
- [ ] **(C)** The temperature must be above 10,000 K.
- [ ] **(D)** All atoms must have zero momentum.

In [36]:
ws.render_section('q20_28')

**Q28 · medium** · 💡

If $n_0 = 10,000$ configurations are collected in state 0 and $n_1 = 1,000$ configurations in state 1, what is the shift term $\ln(n_0 / n_1)$ in the BAR equation? (Enter to 2 decimal places).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\ln(10,000 / 1,000) = \ln(10)$.

</div>
</details>

_Numeric answer required._

In [37]:
ws.render_section('q20_29')

**Q29 · hard** · 💡

How is the non-equilibrium Jarzynski equality $\langle e^{-\beta W} \rangle = e^{-\beta \Delta F}$ related to the equilibrium free energy difference?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall Jarzynski's remarkable 1997 equality for non-equilibrium pulling trajectories.

</div>
</details>

- [ ] **(A)** It states that the average work $\langle W \rangle$ always equals $\Delta F$ regardless of pulling speed.
- [ ] **(B)** It relates the exponential average of non-equilibrium work $W$ performed during finite-time irreversible processes to the equilibrium free energy difference $\Delta F$.
- [ ] **(C)** It only holds for infinitely slow, quasi-static processes where $W = \Delta F$.
- [ ] **(D)** It is an approximation valid only for linear harmonic springs.

In [38]:
ws.render_section('q20_30')

**Q30 · medium** · 💡

Which of the following statements about Crooks Fluctuation Theorem are TRUE? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Remember that Crooks theorem is a non-equilibrium relation for forward and time-reversed protocols.

</div>
</details>

- [ ] **(A)** It applies to microscopic work distributions $P_F(W)$ in forward driving and $P_R(-W)$ in reverse driving: $P_F(W)/P_R(-W) = e^{\beta(W - \Delta F)}$.
- [ ] **(B)** Integrating Crooks relation over all work values directly yields the Jarzynski equality.
- [ ] **(C)** It requires the system to remain in equilibrium at every intermediate instant of the protocol.
- [ ] **(D)** The forward and reverse work distributions intersect at $W = \Delta F$.

In [39]:
ws.render_section('sec_umbrella_and_wang_landau')

### Overcoming Free Energy Barriers: Umbrella Sampling

When physical processes (such as chemical reactions, conformational transitions, or nucleation) involve large free energy barriers $\Delta F^* \gg k_B T$, standard simulations remain trapped in local free energy minima. Torrie and Valleau (1977) introduced **Umbrella Sampling**, where an artificial biasing potential $W(\xi)$ is added along a reaction coordinate $\xi$:

$$V_{\text{biased}}(\mathbf{r}^N) = V(\mathbf{r}^N) + W(\xi(\mathbf{r}^N))$$

Typically, a series of harmonic windows $W_i(\xi) = \frac{1}{2}k(\xi - \xi_i)^2$ restrain the system across the entire reaction coordinate. The unbiased probability distribution $P_i(\xi)$ in window $i$ is recovered by unbiasing the observed histogram $P_i^{\text{biased}}(\xi)$:

$$P_i(\xi) = P_i^{\text{biased}}(\xi) e^{\beta W_i(\xi)} e^{-\beta F_i}$$

The **Weighted Histogram Analysis Method (WHAM)** combines all overlapping biased histograms into a single optimal potential of mean force (PMF) $A(\xi) = -k_B T \ln P(\xi)$.

### The Wang-Landau Algorithm

Rather than sampling according to Boltzmann weights, Wang and Landau (2001) designed an algorithm that samples configuration space with probability inversely proportional to the **density of states** $g(E)$:

$$P_{\text{target}}(E) \propto \frac{1}{g(E)}$$

If this target distribution is sampled, the resulting energy histogram $H(E) \propto g(E) P(E) \propto 1$ is completely **flat**, allowing the simulation to perform a random walk across all energy levels and cross arbitrary energy barriers freely!

**Wang-Landau Algorithm Steps:**
1. Initialize density of states $g(E) = 1$ (or $\ln g(E) = 0$), histogram $H(E) = 0$, and modification factor $f = f_0 = e^1 \approx 2.71828$.
2. Propose a move from state $A$ with energy $E_A$ to state $B$ with energy $E_B$. Accept with probability:
$$P(A \to B) = \min\left( 1, \frac{g(E_A)}{g(E_B)} \right)$$
3. Whichever state $E$ is current (accepted or rejected), update the density of states and histogram:
$$\ln g(E) \leftarrow \ln g(E) + \ln f, \qquad H(E) \leftarrow H(E) + 1$$
4. Check for histogram flatness (e.g. all $H(E) \ge 0.80 \langle H \rangle$). When flat, refine $f$:
$$\ln f_{k+1} = \frac{1}{2}\ln f_k \iff f_{k+1} = \sqrt{f_k}$$
Reset $H(E) = 0$ for all $E$, and continue until $\ln f < 10^{-8}$.


In [40]:
ws.render_section('concept_wang_landau_flat_histogram')

### Interactive Concept: Wang-Landau Density of States & Flat Histogram

Explore the iterative convergence of the Wang-Landau algorithm.
Step through the refinement stages (`wl_stage` from 1 to 15), adjust the flatness criterion, and toggle between the microcanonical entropy $S(E)/k_B = \ln g(E)$ and the energy histogram $H(E)$.

**Key Insight:** By penalizing frequently visited energies through $\ln g(E) \leftarrow \ln g(E) + \ln f$, Wang-Landau drives the walker into rarely explored high-barrier regions until the energy histogram is flat. Halving $\ln f$ at each stage ensures rigorous asymptotic convergence to the true density of states $g(E)$, from which canonical thermodynamics at all temperatures can be computed with a single run!

In [41]:
ws.render_section('q20_31')

**Q31 · easy** · 💡

What is the primary purpose of applying a biasing potential $W(\xi)$ in Umbrella Sampling?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about why standard Boltzmann sampling fails when barriers exceed several $k_B T$.

</div>
</details>

- [ ] **(A)** To cool the system down to absolute zero.
- [ ] **(B)** To overcome high free energy barriers along a reaction coordinate $\xi$ and force uniform sampling of transition states and rare configurations.
- [ ] **(C)** To remove the kinetic energy from Newton's equations.
- [ ] **(D)** To convert molecular dynamics into quantum mechanics.

In [42]:
ws.render_section('q20_32')

**Q32 · medium** · 💡

In Umbrella Sampling, once a biased histogram $P_i^{\text{biased}}(\xi)$ is measured in window $i$ with harmonic bias $W_i(\xi) = \frac{1}{2}k(\xi - \xi_i)^2$, how is the unbiased probability $P_i(\xi)$ reconstructed?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since the biased distribution is $P_i^{\text{biased}}(\xi) \propto P_i(\xi) e^{-\beta W_i(\xi)}$, invert this relation.

</div>
</details>

- [ ] **(A)** $P_i(\xi) = P_i^{\text{biased}}(\xi) e^{-\beta W_i(\xi)}$
- [ ] **(B)** $P_i(\xi) = P_i^{\text{biased}}(\xi) e^{+\beta W_i(\xi)} e^{-\beta F_i}$
- [ ] **(C)** $P_i(\xi) = [P_i^{\text{biased}}(\xi)]^2$
- [ ] **(D)** $P_i(\xi) = \frac{d}{d\xi} P_i^{\text{biased}}(\xi)$

In [43]:
ws.render_section('q20_33')

**Q33 · medium** · 💡

What is the acceptance probability $P(A \to B)$ in the Wang-Landau algorithm for a proposed trial move from energy $E_A$ to energy $E_B$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

The target distribution is $1/g(E)$. Apply detailed balance: $\pi(A) P(A \to B) = \pi(B) P(B \to A)$.

</div>
</details>

- [ ] **(A)** $\min\left( 1, e^{-\beta(E_B - E_A)} \right)$ (Metropolis)
- [ ] **(B)** $\min\left( 1, \frac{g(E_A)}{g(E_B)} \right)$
- [ ] **(C)** $\min\left( 1, \frac{g(E_B)}{g(E_A)} \right)$
- [ ] **(D)** $1.0$ unconditionally

In [44]:
ws.render_section('q20_34')

**Q34 · easy** · 💡

In the Wang-Landau update rule $\ln g(E) \leftarrow \ln g(E) + \ln f$, what happens to the modification factor $f$ when the energy histogram $H(E)$ satisfies the flatness criterion?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How is $f$ refined in each stage of Wang-Landau?

</div>
</details>

- [ ] **(A)** $f$ is doubled ($f_{k+1} = 2 f_k$)
- [ ] **(B)** The log modification factor is halved: $\ln f_{k+1} = \frac{1}{2}\ln f_k \implies f_{k+1} = \sqrt{f_k}$, and $H(E)$ is reset to zero
- [ ] **(C)** $f$ is set to zero immediately and the simulation terminates
- [ ] **(D)** $f$ remains unchanged forever

In [45]:
ws.render_section('q20_35')

**Q35 · easy** · 💡

Starting with an initial Wang-Landau modification factor $f_0 = e^1$ (so $\ln f_0 = 1.0$), what is the value of $\ln f$ after $k = 4$ successful flatness refinement stages?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each stage divides $\ln f$ by 2: evaluate $(1/2)^4$.

</div>
</details>

_Numeric answer required._

In [46]:
ws.render_section('q20_36')

**Q36 · medium** · 💡

How many refinement stages $k$ are required to reduce $\ln f$ from $1.0$ to below $10^{-6}$ (i.e. $(1/2)^k < 10^{-6}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve $2^k > 10^6$: $k > 6 \times \log_2(10) \approx 6 \times 3.322$.

</div>
</details>

_Numeric answer required._

In [47]:
ws.render_section('q20_37')

**Q37 · medium** · 💡

Once the density of states $g(E)$ has been accurately determined by Wang-Landau, how can the canonical partition function $Z(T)$ and average energy $\langle E \rangle_T$ be computed at ANY temperature $T$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the microcanonical-to-canonical transformation $Z(\beta) = \int g(E) e^{-\beta E} dE$.

</div>
</details>

- [ ] **(A)** A new Wang-Landau simulation must be run for every temperature.
- [ ] **(B)** They can be evaluated immediately by numerical quadrature: $Z(T) = \sum_E g(E) e^{-E / k_B T}$ and $\langle E \rangle_T = \frac{1}{Z(T)} \sum_E E g(E) e^{-E / k_B T}$ without any additional simulations!
- [ ] **(C)** By differentiating $g(E)$ with respect to pressure.
- [ ] **(D)** The density of states cannot be used to compute canonical averages.

In [48]:
ws.render_section('q20_38')

**Q38 · hard** · 💡

In the microcanonical ensemble, the microcanonical entropy is defined as $S(E) = k_B \ln g(E)$. How is the microcanonical temperature $T(E)$ defined from $S(E)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the fundamental thermodynamic definition of temperature $1/T = (\partial S / \partial E)_{V, N}$.

</div>
</details>

- [ ] **(A)** $T(E) = \frac{E}{k_B}$
- [ ] **(B)** $\frac{1}{T(E)} = \frac{\partial S(E)}{\partial E} = k_B \frac{\partial \ln g(E)}{\partial E}$
- [ ] **(C)** $T(E) = \frac{\partial^2 S}{\partial E^2}$
- [ ] **(D)** $T(E) = \sqrt{S(E)}$

In [49]:
ws.render_section('q20_39')

**Q39 · medium** · 💡

Which of the following flat-histogram and enhanced sampling methods belong to the generalized-ensemble family? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look for methods that adaptively construct a non-Boltzmann bias to flatten energy or collective variable distributions.

</div>
</details>

- [ ] **(A)** Wang-Landau sampling
- [ ] **(B)** Multicanonical Monte Carlo (MUCA / Berg & Neuhaus)
- [ ] **(C)** Metadynamics (Laio & Parrinello)
- [ ] **(D)** Standard microcanonical NVE Molecular Dynamics

In [50]:
ws.render_section('q20_40')

**Q40 · medium** · 💡

In the concept cell, at early refinement stages ($k=1$), what does the energy histogram $H(E)$ look like before the modification factor is reduced?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Observe the histogram plot in the early stages when $f$ is large.

</div>
</details>

- [ ] **(A)** It is perfectly flat across all energies.
- [ ] **(B)** It shows significant statistical fluctuations and uneven coverage until the flatness criterion is met and $f$ is decreased.
- [ ] **(C)** It is zero everywhere.
- [ ] **(D)** It is an inverted parabola.

In [51]:
ws.render_section('q20_41')

**Q41 · hard** · 💡

If a system exhibits a strong first-order phase transition, what distinctive signature appears in the microcanonical entropy $S(E) = k_B \ln g(E)$ curve?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What does a barrier between two coexistence phases imply for the concavity of $S(E)$ in a finite system?

</div>
</details>

- [ ] **(A)** $S(E)$ is strictly linear with slope 0.
- [ ] **(B)** $S(E)$ has a region of positive curvature ($d^2 S / dE^2 > 0$), producing a convex 'dip' that corresponds to a negative microcanonical heat capacity ($C_V < 0$) in finite systems.
- [ ] **(C)** $S(E)$ drops to negative infinity.
- [ ] **(D)** $S(E)$ becomes discontinuous.

In [52]:
ws.render_section('q20_42')

**Q42 · medium** · 💡

A reaction has a free energy barrier of $\Delta F^* = 10.0\text{ kcal/mol}$ at $T = 300\text{ K}$ ($k_B T = 0.596\text{ kcal/mol}$). By what factor is the transition state probability reduced relative to the reactant minimum ($e^{-\Delta F^* / k_B T}$)? Enter the order of magnitude exponent $m$ where the factor is $10^{-m}$ (i.e. $m = \frac{\Delta F^*}{k_B T \ln 10}$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $10.0 / (0.596 \times 2.3026)$.

</div>
</details>

_Numeric answer required._

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()